# Human-AI Collaboration with MOSAIC

**Notebook 2 of 2.** MOSAIC is a grid-based **search-and-rescue** task: a building on fire, victims trapped in rooms,
lava on the floor, some doors locked. A human, an AI, or both together must save the victims before time runs out.
We build the task, look inside it, play it, and record the player's **actions and eye gaze** with LSL into an XDF file,
using the same helpers as Notebook 1 (SHASTA, human-swarm interaction). Notebook 1 is not in this repository;
the helper file it shares with this notebook, `lsl_tools.py`, is in this folder, so this notebook runs on its own.

| Step | Topic |
|---|---|
| **1** | Check the environment |
| **2** | The config file and the search-and-rescue environment |
| **3** | Actions: move the rescuer |
| **4** | What the agent observes |
| **5** | Agents: random vs a scripted rescuer |
| **6** | How the config changes the game |
| **7** | The human interface, drawn off-screen |
| **8** | Record game state and eye gaze to XDF |
| **9** | Load the XDF and analyse gaze against the game |

Select the **Python (smc)** kernel. Keep `config.yaml` and `lsl_tools.py` in the same folder as this notebook.
To run it on your own laptop instead of the hub, add five packages to the environment MOSAIC is installed in:
`python -m pip install matplotlib scipy pylsl pyxdf notebook`.

## Step 1: Check the environment

As in Notebook 1, the hub has no screen, so pygame must draw on SDL's `dummy` video driver. Set it **before** importing pygame.

In [ ]:
import os, sys
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"
sys.path.insert(0, os.getcwd())

In [ ]:
import importlib

for name in ["numpy", "matplotlib", "scipy", "pygame", "pygame_gui", "minigrid", "gymnasium", "mosaic", "yaml", "pylsl", "pyxdf", "lsl_tools"]:
    try:
        module = importlib.import_module(name)
        version = getattr(module, "__version__", None) or getattr(getattr(module, "version", None), "ver", "")
        print(f"ok       {name:11s} {version}")
    except Exception as exc:
        print(f"MISSING  {name:11s} {type(exc).__name__}: {exc}")

**Checkpoint.** Every line says `ok`. `lsl_tools` is the helper file in this folder, shared with Notebook 1.

## Step 2: The config file and the search-and-rescue environment

MOSAIC is built on **MiniGrid**. `build_sar_env` creates the whole task: a building of `num_rows x num_cols` rooms, each
`room_size` tiles wide. Three **placers** decide what goes where:

| Placer | Decides |
|---|---|
| `VictimPlacer(num_real_victims=n)` | how many victims per room, and where |
| `LavaPlacer(lava_per_room=n)` | where the lava hazards are |
| `LockedRoomPlacer(locked_room_prob=p)` | which doors are locked (their keys are hidden in other rooms) |

`VictimPlacer` places **real victims only**. The study runner (`python -m experiment.main`) uses `LavaRiskVictimPlacer`, which also places
**decoys**: look-alike victims that cost a point. This notebook's building has none.

`render_mode="rgb_array"` means the environment returns an **image** from `env.render()`. Nothing opens a window.

First the **config file**: a short YAML text file next to this notebook. Its `game` section uses the same key names as the real study's
`configs/experiment.yaml`, with a smaller building so that everything runs in seconds. The notebook builds the task from it, and Step 6 changes
its values one at a time to show what each one does.

In [ ]:
print(open("config.yaml").read())

In [ ]:
import random

import numpy as np
import matplotlib.pyplot as plt

from mosaic.sar.env import build_sar_env
from mosaic.sar.placers import VictimPlacer, LavaPlacer, LockedRoomPlacer

import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)
GAME = config["game"]
print("game config:", GAME)

def make_env(game=None, **changes):
    """Build the task from the ``game`` section of the config (optionally with some values changed)."""
    g = {**(game or GAME), **changes}
    return build_sar_env(
        screen_size=g["screen_size"], num_rows=g["num_rows"], num_cols=g["num_cols"],
        room_size=g["room_size"], tile_size=g["tile_size"],
        victim_placer=VictimPlacer(num_real_victims=g["num_real_victims"]),
        lava_placer=LavaPlacer(lava_per_room=g["lava_per_room"]),
        locked_room_placer=LockedRoomPlacer(locked_room_prob=g["locked_room_prob"]),
    )

def reset_level(env, seed):
    """Start level ``seed``. The placers draw from Python's ``random`` module, so seed it together with the environment."""
    random.seed(seed)
    return env.reset(seed=seed)

env = make_env()
obs, info = reset_level(env, 0)
print("mission:", obs["mission"])
print("max steps:", env.max_steps)

plt.figure(figsize=(6, 6)); plt.imshow(env.render()); plt.axis("off"); plt.title("seed 0: what env.render() returns")
plt.show()

A different `seed` builds a different building. The seed makes experiments **reproducible**: everyone who uses seed 0 plays the same level.
`env.reset(seed=0)` on its own is not enough: it fixes the rooms and doors, but MOSAIC's placers put the victims, lava and keys down with
Python's `random` module. `reset_level` seeds both.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, seed in zip(axes, (1, 2, 3)):
    reset_level(env, seed)
    ax.imshow(env.render()); ax.set_title(f"seed {seed}"); ax.axis("off")
plt.show()

**Checkpoint.** Four different levels. The task parameters (rooms, victims, lava, locks) all come from `config.yaml`; change them there and look again.

## Step 3: Actions: move the rescuer

The rescuer has seven discrete actions:

| Action | Effect |
|---|---|
| `left`, `right` | turn 90 degrees |
| `forward` | move one tile ahead |
| `pickup` | rescue the victim in front |
| `drop`, `toggle`, `done` | drop an item, open a door, finish |

`env.step(action)` returns the Gymnasium tuple `(observation, reward, terminated, truncated, info)`.

In [ ]:
print(list(env.actions))

reset_level(env, 0)
script = [env.actions.right] + [env.actions.forward] * 5 + [env.actions.left] + [env.actions.forward] * 4
frames, rewards = [env.render()], []
for action in script:
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
frames.append(env.render())
print("steps:", len(script), " total reward:", sum(rewards), " info:", info)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, frame, title in zip(axes, frames, ("before", f"after {len(script)} actions")):
    ax.imshow(frame); ax.set_title(title); ax.axis("off")
plt.show()

Rewards come from **events**: a rescued victim is `+1` by default (`RescueRewards` sets the values), a decoy or a dead victim is negative.
The `info["events"]` list says what happened on each step.

**Checkpoint.** The rescuer has moved. Next we look at the numbers behind the picture.

## Step 4: What the agent observes

`env.step` and `env.reset` also return an **observation**: a dictionary with everything a program needs to understand the game:
the agent's position and direction, the whole building as a **grid of integer codes**, how many victims are saved and how much
health each victim has left.

In [ ]:
obs, _ = reset_level(env, 0)
for key, value in obs.items():
    shown = value if not isinstance(value, (list, dict)) else f"{type(value).__name__} of length {len(value)}"
    print(f"{key:18s} {shown}")

The `grid` is a table of codes. `MOSAIC.sar.observations` defines them: `0` empty floor, `1` wall, `4` lava, `5` real victim, `6` decoy (`FAKE_VICTIM`),
`10`-`29` doors (colour, open/locked), `30`+ keys. Let's draw it.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from mosaic.sar import observations as O

grid = np.array(obs["grid"])
category = np.zeros_like(grid)
category[grid == O.EMPTY] = 0
category[grid == O.WALL] = 1
category[grid == O.LAVA] = 2
category[grid == O.VICTIM] = 3
category[grid == O.FAKE_VICTIM] = 4
category[(grid >= O.DOOR_BASE) & (grid < O.KEY_BASE)] = 5
category[grid >= O.KEY_BASE] = 6
names = ["floor", "wall", "lava", "victim", "decoy", "door", "key"]
colors = ["#eeeeee", "#222222", "#ee6a1c", "#2ca02c", "#d2b92b", "#1f77b4", "#9467bd"]

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(category, cmap=ListedColormap(colors), norm=BoundaryNorm(np.arange(-0.5, 7.5), 7))
ax.plot(obs["agent_x"], obs["agent_y"], "r>", markersize=14, label="rescuer")
for i, (n, c) in enumerate(zip(names, colors)):
    ax.plot([], [], "s", color=c, label=n)
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5)); ax.set_title("The same level as a grid of codes")
plt.show()
print("victim health:", obs["victim_health"])

**Checkpoint.** The grid matches the picture from Step 2: walls around rooms, doors between them, victims in green.
`victim_health` shows each victim's remaining health. Here it stays at `1.0`: health drains only in the study runner
(`TunedPickupVictimEnv` in `src/experiment/`), where a victim that reaches zero dies.

## Step 5: Agents: random vs a scripted rescuer

Because MOSAIC is a Gymnasium environment, any program can play it. We compare two:

- **random**: picks a random action every step.
- **rescuer**: plans a shortest path (breadth-first search) over the grid to the nearest reachable victim, opens unlocked doors on the way,
  and picks the victim up.

In [ ]:
from collections import deque

DIRS = [(1, 0), (0, 1), (-1, 0), (0, -1)]       # agent_dir 0..3: right, down, left, up

def passable(env, x, y, goal):
    if (x, y) == goal:
        return True
    obj = env.grid.get(x, y)
    if obj is None:
        return True
    if obj.type == "door":
        return not obj.is_locked
    return False                                  # walls, lava, keys, other victims

def plan(env, goal):
    """Shortest path of cells from the agent to ``goal`` (breadth-first search), or None."""
    start = tuple(int(v) for v in env.agent_pos)
    queue, came = deque([start]), {start: None}
    while queue:
        cell = queue.popleft()
        if cell == goal:
            path = []
            while cell != start:
                path.append(cell); cell = came[cell]
            return path[::-1]
        for dx, dy in DIRS:
            nxt = (cell[0] + dx, cell[1] + dy)
            if nxt not in came and 0 <= nxt[0] < env.width and 0 <= nxt[1] < env.height and passable(env, *nxt, goal):
                came[nxt] = cell; queue.append(nxt)
    return None

def rescuer(env, rng):
    """Next action of the scripted rescuer, or None when no victim can be reached."""
    best = None
    for victim in env.get_all_victims():
        path = plan(env, tuple(int(v) for v in victim.cur_pos))
        if path and (best is None or len(path) < len(best)):
            best = path
    if best is None:
        return None
    nx, ny = best[0]
    ax, ay = (int(v) for v in env.agent_pos)
    want = DIRS.index((nx - ax, ny - ay))
    if env.agent_dir != want:                                   # turn towards the next cell
        return env.actions.right if (want - env.agent_dir) % 4 == 1 else env.actions.left
    front = env.grid.get(nx, ny)
    if front is not None and front.type == "door" and not front.is_open:
        return env.actions.toggle
    return env.actions.pickup if len(best) == 1 else env.actions.forward

def play(policy, seed, max_steps=600):
    env = make_env(); reset_level(env, seed)
    rng = np.random.default_rng(seed)
    for step in range(max_steps):
        action = policy(env, rng)
        if action is None:
            break
        _, _, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break
    status = env.get_mission_status()
    return {"saved": status["saved_victims"], "of": status["saved_victims"] + status["remaining_victims"], "steps": step + 1}

random_policy = lambda env, rng: env.actions(int(rng.integers(0, 7)))

In [ ]:
import pandas as pd

rows = []
for seed in range(10):
    for name, policy in (("random", random_policy), ("rescuer", rescuer)):
        rows.append({"seed": seed, "agent": name, **play(policy, seed)})
results = pd.DataFrame(rows)
print(results.groupby("agent")[["saved", "steps"]].mean().round(1))

fig, ax = plt.subplots(figsize=(7, 4))
results.pivot(index="seed", columns="agent", values="saved").plot.bar(ax=ax)
ax.set_ylabel("victims saved (of ~8)"); ax.set_title("Same ten levels, two agents")
plt.tight_layout(); plt.show()

**Checkpoint.** The scripted rescuer saves far more victims than the random agent, but usually not **all**: the victims left over are ones it cannot reach, typically behind **locked
doors** (it never looks for the keys) or blocked by lava. A human, or a smarter AI, has to solve those. That is where human-AI collaboration begins.

## Step 6: How the config changes the game

We keep the **same eight seeds** (0 to 7) and change **one config value at a time**. The scripted rescuer plays every level and we measure how many
victims it saves, how many steps it needs, and how many steps the game allows (`max_steps`, which grows with the size of the building).
Because the seeds and the player are the same, the numbers are the same on every run, and a difference between rows comes from the value we changed
(including where it moves the victims, lava and keys).

In [ ]:
import pandas as pd

def play_level(game, seed, max_steps=1500):
    env = make_env(game); reset_level(env, seed)
    rng = np.random.default_rng(seed)
    for step in range(max_steps):
        action = rescuer(env, rng)
        if action is None:
            break
        _, _, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break
    status = env.get_mission_status()
    total = status["saved_victims"] + status["remaining_victims"]
    return {"victims": total, "saved": status["saved_victims"], "steps": step + 1, "max_steps": env.max_steps}

def sweep(changes, seeds=range(8)):
    """``changes`` is a list of dicts of config values to override. One result row per dict."""
    rows = []
    for change in changes:
        game = {**GAME, **change}
        res = pd.DataFrame([play_level(game, s) for s in seeds])
        rows.append({**change, "victims": res["victims"].mean(), "saved": res["saved"].mean(),
                     "saved_%": 100 * res["saved"].sum() / res["victims"].sum(), "steps": res["steps"].mean(),
                     "max_steps": res["max_steps"].mean()})
    return pd.DataFrame(rows).round(1)

print("baseline (config.yaml as it is):")
display(sweep([{}]))

### 1. `num_real_victims`: how many people to save

More victims per room means a bigger task: the rescuer needs more steps and the game allows more of them (`max_steps`). The share saved stays about the same.

In [ ]:
victims_table = sweep([{"num_real_victims": v} for v in (1, 2, 4)])
display(victims_table)

### 2. `lava_per_room`: hazards

Lava is impassable for the scripted rescuer, so more lava means longer detours (more steps) and can cut off a victim. Here the effect is small.

In [ ]:
lava_table = sweep([{"lava_per_room": v} for v in (0, 1, 3, 6)])
display(lava_table)

### 3. `locked_room_prob`: locked doors

A locked room needs its key. The scripted rescuer never looks for keys, so the more rooms are locked, the fewer victims it can reach. At a high probability the share saved collapses.

In [ ]:
locked_table = sweep([{"locked_room_prob": v} for v in (0.0, 0.35, 0.8)])
display(locked_table)

### 4. The size of the building: `num_rows` and `num_cols`

A bigger building has more rooms, so more victims, longer walks and a larger time budget (`max_steps`). The player stops after at most 1500 steps.
A building of only 1 x 1 rooms cannot be generated (the level generator never finishes), so the smallest size is 2 x 2.

In [ ]:
size_table = sweep([{"num_rows": n, "num_cols": n} for n in (2, 3)])
display(size_table)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, (table, key, title) in zip(axes, [(lava_table, "lava_per_room", "Lava"), (locked_table, "locked_room_prob", "Locked doors"),
                                          (victims_table, "num_real_victims", "Victims per room")]):
    ax.bar(table[key].astype(str), table["saved_%"], color="#4c78a8")
    ax.set_xlabel(key); ax.set_ylabel("victims saved (%)"); ax.set_ylim(0, 105); ax.set_title(title)
plt.tight_layout(); plt.show()

**Checkpoint.** One text file controls how hard the task is: how many people there are to save, how much lava and how many locked doors stand
in the way, and how large the building (and the time budget) is. Several keys (`max_time`, `prompt_type`, `llm_nudge_interval`, `fullscreen`) do not change the
simulation at all but control **what the participant sees and gets**. The next step shows one of them. They work here because this notebook passes its `game` section to
`SAREnvGUI` as the whole config. `python -m experiment.main` passes the whole of `configs/experiment.yaml`, so there these keys must be
**top-level** lines, not under `game:`.

*Try it:* change a value in `config.yaml`, for example `num_real_victims: 4` or `locked_room_prob: 0.8`, and re-run from Step 2.

## Step 7: The human interface, drawn off-screen

`SAREnvGUI` is the pygame window a participant plays in: the game view on the left, an information panel (victims saved, time left)
and a chat panel for the AI teammate on the right. `gui.user` holds the game state; `gui.user.step(action)` plays one action,
exactly like a key press. We draw the window on pygame's dummy driver and read its surface as an image.

In [ ]:
import pygame
from mosaic.gui.main import SAREnvGUI

def gui_frame(gui):
    """The GUI window as a (height, width, 3) image."""
    return np.asarray(pygame.surfarray.array3d(gui.window)).swapaxes(0, 1)

env = make_env()
gui = SAREnvGUI(env, config=GAME)
gui.reset()
gui.user.obs, _ = reset_level(env, 0)

gui.render(gui.user.get_frame())
print("window size:", gui.window_size, "  game view:", gui.game_size, "px, panel width:", gui.panel_width, "px")
plt.figure(figsize=(12, 8)); plt.imshow(gui_frame(gui)); plt.axis("off"); plt.title("The participant's screen")
plt.show()

Now let the scripted rescuer act through the GUI, as if the participant pressed keys, and look again.

In [ ]:
rng = np.random.default_rng(0)
for _ in range(60):
    action = rescuer(gui.user.env, rng)
    if action is None:
        break
    gui.user.step(action)
    gui.render(gui.user.get_frame())

plt.figure(figsize=(12, 8)); plt.imshow(gui_frame(gui)); plt.axis("off"); plt.title("After 60 actions")
plt.show()
print("rescued so far:", gui.user.env.get_mission_status())

The config also reaches the screen. `max_time` is the minutes on the participant's clock, so changing it changes the **information panel** the participant reads:

In [ ]:
panels = {}
for minutes in (5, 1):
    env_m = make_env()
    gui_m = SAREnvGUI(env_m, config={**GAME, "max_time": minutes})
    gui_m.reset(); gui_m.render(gui_m.user.get_frame())
    panels[minutes] = gui_frame(gui_m)[: gui_m.game_size // 2, gui_m.game_size:]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, (minutes, img) in zip(axes, panels.items()):
    ax.imshow(img); ax.axis("off"); ax.set_title(f"max_time: {minutes}")
plt.tight_layout(); plt.show()

**Checkpoint.** The panel on the right counts rescued victims and shows the clock, and the clock follows the config. This is the screen where a participant's gaze goes.

## Step 8: Record game state and eye gaze to XDF

The recording has the same three parts as in Notebook 1:

1. A **state stream** from the game: one marker per action, with the fields MOSAIC's own `SARGameTrial` streams
   (`agent_x`, `agent_y`, `step_count`, `saved_victims`, `action`, `reward`), plus an `event` marker whenever something happens (a rescue).
2. A **gaze stream** from the eye tracker: here synthetic, and in the real study the Tobii.
3. A **recorder** that writes both to one XDF file.

### Eye tracking in MOSAIC

MOSAIC's own study code already has eye tracking (`src/experiment/sensors/eye_tracker/`):

| Piece | What it does |
|---|---|
| `TobiiEyeTracker` | connects to the Tobii, and publishes a **`Gaze`** LSL stream: 9 channels at 60 Hz (`device_timestamp`, `avg_gaze_point_x/y` in pixels, `avg_pupil_diam`, eye position x/y/z, `avg_eye_distance`, `eye_validities`) |
| calibration | the Tobii is calibrated before the experiment, with an optional recalibration between trials (`between_trail_recalibrate_eye_tracker` in `configs/experiment.yaml`) |
| `SARGameTrial` | streams the game state through LSL at every frame, next to the gaze |

If no Tobii is detected, the game simply runs without eye tracking. The study code needs the `ixp` package, `psychopy` and `tobii-research`, so it runs on the study computer.
`tobii_to_lsl.py` in this folder is a small stand-alone alternative that publishes the **same 9-channel `Gaze` stream** straight from the Tobii SDK.

In the real study, **LabRecorder** records the `Gaze` stream and the game stream together on the study computer into one XDF file, and the analysis in Step 9 does not change.
Here the `eye_tracker.source` key in `config.yaml` chooses the gaze source: `synthetic` (made-up gaze, for a demo without hardware) or `live` (record whatever `Gaze` stream is on the network).

### Areas of interest (AOIs)

We split the screen into the **game view**, the **information panel** and the **chat panel**, and record those rectangles in the file,
so the analysis knows where the participant could look.

In [ ]:
import time
from lsl_tools import MarkerOutlet, SyntheticGaze, Recorder

XDF_PATH = "mosaic_session.xdf"
SESSION_SECONDS = config["session"]["seconds"]      # from config.yaml
ACT_EVERY = config["session"]["act_every"]         # one game action every N frames (10 = about 3 per second)

env = make_env()
gui = SAREnvGUI(env, config=GAME)
gui.reset()
user = gui.user
user.obs, _ = reset_level(env, 0)

W, H = gui.window_size
AOIS = {"game": [0, 0, gui.game_size, gui.game_size],
        "info": [gui.game_size, 0, gui.panel_width, gui.game_size // 2],
        "chat": [gui.game_size, gui.game_size // 2, gui.panel_width, gui.game_size // 2]}
print("AOIs (x, y, width, height):", AOIS)

markers = MarkerOutlet("MOSAIC-State")
USE_SYNTHETIC = config["eye_tracker"]["source"] == "synthetic"
gaze = SyntheticGaze(width=W, height=H).start() if USE_SYNTHETIC else None      # "live": use the Gaze stream already on the network

def look(x, y):
    """Steer the synthetic gaze (does nothing when a real eye tracker is the source)."""
    if gaze is not None:
        gaze.look_at(x, y)
recorder = Recorder(XDF_PATH, stream_types=("Markers", "Gaze")).start()
print("recording:", recorder.streams)

The loop runs the game in real time at 30 frames per second. The synthetic gaze **scans around the agent** in the game view,
**glances at the information panel right after every rescue** to check the counter, and looks at the clock every few seconds.
That is a stand-in for how a real participant behaves, and it is the pattern the analysis will look for.

In [ ]:
rng = np.random.default_rng(0)
markers.push("session_start", screen=[W, H], aois=AOIS, synthetic_gaze=USE_SYNTHETIC)

glance_until, next_clock_check, frame, start = 0, 5.0, 0, time.time()
last_state = None
while time.time() - start < SESSION_SECONDS:
    now = time.time() - start

    if frame % ACT_EVERY == 0:
        action = rescuer(user.env, rng)
        if action is None:
            break
        user.step(action)
        obs = user.obs
        markers.push("state", step_count=obs["step_count"], agent_x=obs["agent_x"], agent_y=obs["agent_y"],
                     saved_victims=obs["saved_victims"], remaining_victims=obs["remaining_victims"],
                     action=int(action), reward=float(user.last_reward))
        for event in user.last_info.get("events", []):
            markers.push("rescue" if event.get("type") == "victim_rescued" else "event", **event)
            glance_until = now + 0.6                                   # check the counter

    if now < glance_until:
        info = AOIS["info"]; look(info[0] + 0.5 * info[2], info[1] + 0.25 * info[3])
    elif now > next_clock_check:
        info = AOIS["info"]; look(info[0] + 0.5 * info[2], info[1] + 0.8 * info[3])
        if now > next_clock_check + 0.6:
            next_clock_check = now + 5.0
    elif frame % 12 == 0:
        center = AOIS["game"][2] / 2
        look(center + rng.normal(0, 90), center + rng.normal(0, 90))   # scan around the agent

    gui.render(user.get_frame())              # also paces the loop at 30 frames per second
    frame += 1
    if frame == 150:
        plt.imsave("mosaic_frame.png", gui_frame(gui))

markers.push("session_end", saved=user.obs["saved_victims"], frames=frame)
saved_path = recorder.stop()
if gaze is not None:
    gaze.stop()
print(f"{frame} frames, {user.obs['saved_victims']} victims saved")
print("saved", saved_path, f"({os.path.getsize(saved_path) / 1024:.0f} KB)")

**Checkpoint.** An `.xdf` file with two streams: the game (states and rescues) and the gaze.

## Step 9: Load the XDF and analyse gaze against the game

Set `XDF_PATH` to a real LabRecorder file to analyse a real session. For gaze in real screen coordinates the game window must fill the screen
(MOSAIC runs fullscreen in the experiment) and the AOIs must be the rectangles for that screen.

In [ ]:
import pandas as pd
from lsl_tools import load_streams, gaze_table, marker_table, detect_fixations, gaze_heatmap

XDF_PATH = "mosaic_session.xdf"            # <- your LabRecorder file for a real recording
streams = load_streams(XDF_PATH)
for s in streams.values():
    span = s["ts"][-1] - s["ts"][0]
    print(f"{s['type']:8s} {s['name']:14s} {len(s['ts']):6d} samples over {span:5.1f} s{'   <-- SYNTHETIC data' if s['synthetic'] else ''}")

gaze_data = gaze_table(streams["Gaze"])
rows = marker_table(streams["Markers"])
t0 = min(gaze_data["t"][0], rows[0]["t"])
gt = gaze_data["t"] - t0
for r in rows: r["t"] -= t0

config = next(r for r in rows if r.get("event") == "session_start")
SCREEN, AOIS = tuple(config["screen"]), config["aois"]
states = pd.DataFrame([r for r in rows if r.get("event") == "state"])
rescues = [r for r in rows if r.get("event") == "rescue"]
print(f"{len(states)} game states, {len(rescues)} rescues, gaze valid {gaze_data['valid'].mean():.0%}")

### Where did the participant look? Dwell time per area of interest

In [ ]:
def inside(x, y, rect):
    return (x >= rect[0]) & (x < rect[0] + rect[2]) & (y >= rect[1]) & (y < rect[1] + rect[3])

valid = gaze_data["valid"]
x, y = gaze_data["x"], gaze_data["y"]
share = {name: float(inside(x, y, rect)[valid].mean()) for name, rect in AOIS.items()}
print({k: f"{v:.0%}" for k, v in share.items()})

fixations = detect_fixations(gaze_data["t"], x, y, max_dispersion=40, min_duration=0.10)
heat = gaze_heatmap(x, y, shape=(SCREEN[1], SCREEN[0]))
background = plt.imread("mosaic_frame.png") if os.path.exists("mosaic_frame.png") else None

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(share.keys(), share.values(), color=["#4c78a8", "#f58518", "#54a24b"]); axes[0].set_ylabel("share of valid gaze"); axes[0].set_title("Dwell time by area")
for ax in axes[1:]:
    if background is not None:
        ax.imshow(background, extent=[0, SCREEN[0], SCREEN[1], 0])
    ax.set_xlim(0, SCREEN[0]); ax.set_ylim(SCREEN[1], 0); ax.axis("off")
axes[1].imshow(heat, extent=[0, SCREEN[0], SCREEN[1], 0], cmap="hot", alpha=0.55); axes[1].set_title("Gaze heatmap")
for f in fixations:
    axes[2].scatter(f["x"], f["y"], s=(f["end"] - f["start"]) * 3000, facecolors="none", edgecolors="cyan", linewidths=1.5)
axes[2].set_title(f"{len(fixations)} fixations")
plt.tight_layout(); plt.show()

### Does gaze follow the game events?

After each rescue, does the participant check the information panel? We compare the share of gaze on the panel in the **1.5 s after a rescue**
with the share in all **other** times.

In [ ]:
on_info = inside(x, y, AOIS["info"]) & valid
after = np.zeros(len(gt), dtype=bool)
for r in rescues:
    after |= (gt >= r["t"]) & (gt < r["t"] + 1.5)

print(f"gaze on the info panel within 1.5 s after a rescue: {on_info[after].mean():.0%}")
print(f"gaze on the info panel at other times:              {on_info[~after].mean():.0%}")

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(states["t"], states["saved_victims"], drawstyle="steps-post")
for r in rescues:
    for ax in axes: ax.axvline(r["t"], color="green", alpha=0.5, linestyle="--")
axes[0].set_ylabel("victims saved"); axes[0].set_title("Game progress (dashed = rescue events) and where the eyes were")
axes[1].plot(gt, np.where(on_info, 1, 0), drawstyle="steps-post", color="#f58518")
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["elsewhere", "info panel"]); axes[1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

### The rescuer's path through the building

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(states["agent_x"], states["agent_y"], c=states["t"], cmap="viridis", s=18)
for r in rescues:
    nearest = states.iloc[(states["t"] - r["t"]).abs().argmin()]
    ax.plot(nearest["agent_x"], nearest["agent_y"], "r*", markersize=16)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.set_title("Path (colour = time), red stars = rescues")
plt.show()

## Summary

| What we did | Tool |
|---|---|
| Built a search-and-rescue task from a YAML config and looked at it as an image, a grid and numbers | `config.yaml`, `build_sar_env`, MiniGrid / Gymnasium |
| Changed one config value at a time and measured the effect on difficulty | `sweep`, scripted rescuer |
| Compared a random agent and a planning agent | `env.step`, breadth-first search |
| Drew the participant's screen off-screen | `SAREnvGUI` on the pygame dummy driver |
| Streamed game state and gaze with LSL and saved them as XDF | `pylsl`, `lsl_tools.Recorder` |
| Linked gaze to game events | `pyxdf`, areas of interest, fixations |

Together with Notebook 1 (SHASTA, human-swarm interaction), this is the same pipeline for human-AI interaction: **simulate, interact, record, analyse.**